# HalDetect — NLI rescore under collision-free cache keys (task #16)

**What this fixes.** `relational_verifier_phase3d.py:194` keys every NLI score on

```
nli__{claim[:60]}__{passage[:40]}
```

For the perturbed negatives the 60-character cut lands **mid-number** — `"...Act of
2008 provided $7"` is where four different claims ($700bn, $707bn, $735bn, $7000bn)
become byte-identical. They shared one cache entry, so the first one scored wrote a
value and every later one **read it back as if it were its own**. Twenty-six rows
across both populations are affected; 78 (claim, passage) pairs need real scores.

The visible symptom, to four decimals: `Q5_C04` and `SYN_QUAN_033`–`037` — six claims
with six different values (1.1, 1.2, 1.4, 1.7, 2.2, 11.0) — all carry
`entail=0.6139 contradict=0.1117`. Five of those are gold FALSE and all five were
scored TRUE.

**Why Colab.** BART-large-MNLI SIGBUSes on the author's Mac. That is the *only* step
that needs to leave the Mac: retrieval, chunking, SBERT ranking and the passage filter
were all replayed offline from the recorded cache and are baked into the work-list.
So this notebook does exactly one thing — score 78 pairs — and hands the numbers back.

---

## Three rules for this run

**1. `DEVICE = -1` (CPU).** Not a workaround: `phase3d.get_nli()` hardcodes
`device=-1`, so CPU is the setting the recorded run used. Scoring on GPU would change
the batching and the numeric path, and this rescore is meant to isolate *one*
variable — the cache key. It costs about 3 minutes here, because the passages are
small (mean 453 characters, max 1296). Leave it at `-1`.

**2. Do NOT set `GEMINI_API_KEY` for this run.** There is nothing in this notebook
that calls Gemini, and that is the point. 15 of the 26 rows were originally retrieved
through phase3d's uncached `except: return subject` fallback. Their queries are already
frozen in the work-list. A key in the environment cannot reach them here, but do not
add one — it invites a future reader to assume retrieval was live.

**3. Do not rename the uploaded file.** The next script looks for
`nli_rescore_worklist_<set>.json` by name.

**Runtime:** `Runtime ▸ Change runtime type ▸ CPU` is fine and slightly preferable.
A GPU runtime also works — we simply will not use the GPU.

**Total time:** ~2 min model download + ~3 min scoring.

In [ ]:
# Cell 1 — environment. Nothing is installed unless it is actually missing.
import sys, subprocess, importlib

def have(mod):
    try:
        importlib.import_module(mod); return True
    except ImportError:
        return False

need = [m for m in ("torch", "transformers") if not have(m)]
if need:
    print("installing:", need)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *need])
else:
    print("torch and transformers already present — nothing installed")

import torch, transformers
print("python      :", sys.version.split()[0])
print("torch       :", torch.__version__)
print("transformers:", transformers.__version__)
print("cuda visible:", torch.cuda.is_available(), "(we will NOT use it — see rule 1)")

# Rule 2, asserted rather than trusted.
import os
for var in ("GEMINI_API_KEY", "GOOGLE_API_KEY"):
    if os.environ.get(var):
        print(f"\n!! {var} is set. Nothing here calls Gemini, so it changes no score,")
        print("   but unset it so the run is unambiguous: os.environ.pop('%s')" % var)
GEMINI_KEY_PRESENT = bool(os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY"))

## Cell 2 — upload the work-list

Run the cell, click **Choose Files**, and pick **one** of these from the `current/`
folder of the repo. Start with the first one; the others are supersets and use the
identical notebook.

| file | rows | pairs | what it answers |
|---|---|---|---|
| `nli_rescore_worklist_negatives.json` | 17 | 51 | the collision as #16 states it — do the perturbed negatives get their own scores |
| `nli_rescore_worklist_natural.json` | 9 | 27 | whether the **natural** set is contaminated too, which is the FPR side of the published J |
| `nli_rescore_worklist_both.json` | 26 | 78 | **recommended** — the union, and the only one that can say *which* member of each collision group wrote the entry and which inherited it |

`both` costs ~1 extra minute over `negatives` and answers strictly more, so unless you
want the smallest possible first run, upload `both`.

In [ ]:
# Cell 2 — upload
from google.colab import files
up = files.upload()
WORKLIST = next(n for n in up if n.startswith("nli_rescore_worklist"))
print("using:", WORKLIST)

In [ ]:
# Cell 3 — load and validate. Aborts rather than scoring a work-list it cannot trust.
import json, hashlib, collections

with open(WORKLIST, encoding="utf-8") as fh:
    WL = json.load(fh)
META, ITEMS = WL["meta"], WL["items"]

print("set                :", META["set"])
print("task               :", META["task"])
print("built from         :", META["phase3d_source"])
print("cache entries seen :", META["cache_entries"])
print("reproduction gate  :", META["reproduction_gate"])
print("constants          :", META["constants"])
print("nli spec           :", META["nli"]["candidate_labels"],
      "| template prefix:", repr(META["nli"]["hypothesis_template_prefix"]))

assert META["reproduction_gate"] == "PASS", (
    "The offline reconstruction could not re-derive the recorded scores from the "
    "legacy keys. Until it can, any movement after the rescore is unattributable. "
    "Re-run nli_rescore_prep.py on the Mac and read its gate output.")

PAIRS = [(it["claim"], p["text"], p["fixed_key"]) for it in ITEMS for p in it["passages"]]
keys = [k for _, _, k in PAIRS]
dupe = [k for k, n in collections.Counter(keys).items() if n > 1]
assert not dupe, f"hashed keys are not unique: {dupe[:5]}"

# The hashed key is the whole point of the exercise, so verify it here rather than
# trusting that the Mac computed it correctly.
for claim, passage, key in PAIRS:
    want = "nli2__%s__%s" % (hashlib.sha1(claim.encode()).hexdigest()[:16],
                             hashlib.sha1(passage.encode()).hexdigest()[:16])
    assert key == want, f"key mismatch: {key} != {want}"

# The template is built by string concatenation, so a brace in a claim would break it.
bad = [it["claim_id"] for it in ITEMS if "{" in it["claim"] or "}" in it["claim"]]
assert not bad, f"claims containing braces would corrupt hypothesis_template: {bad}"

print()
print(f"{len(ITEMS)} rows / {len(PAIRS)} pairs, all hashed keys unique and verified")
print("passage chars: min %d  max %d  mean %d" % (
    min(len(p) for _, p, _ in PAIRS), max(len(p) for _, p, _ in PAIRS),
    sum(len(p) for _, p, _ in PAIRS) // len(PAIRS)))
by_pop = collections.Counter(it["population"] for it in ITEMS)
print("by population:", dict(by_pop))
print("collision groups:", len({it["collision_group"] for it in ITEMS}))

In [ ]:
# Cell 4 — load BART-large-MNLI on CPU.
#
# device=-1 is deliberate and matches phase3d.get_nli(), which pins the same value.
# The recorded scores this rescore is compared against were produced on CPU; scoring
# the replacements on GPU would vary the numeric path and confound the one variable
# under test. Change this only if you are deliberately running a different experiment.
DEVICE = -1

from transformers import pipeline

nli = pipeline("zero-shot-classification",
               model="facebook/bart-large-mnli",
               device=DEVICE)
print("loaded facebook/bart-large-mnli on", "CPU" if DEVICE == -1 else f"cuda:{DEVICE}")

In [ ]:
# Cell 5 — score. Same call shape as phase3d:196-200, read from the work-list meta
# rather than retyped, so the two cannot drift apart.
import time

LABELS = META["nli"]["candidate_labels"]
PREFIX = META["nli"]["hypothesis_template_prefix"]

scores, errors = {}, []
t0 = time.time()
for n, (claim, passage, key) in enumerate(PAIRS, 1):
    try:
        res = nli(passage,
                  candidate_labels=LABELS,
                  hypothesis_template=PREFIX + claim)
        got = dict(zip(res["labels"], res["scores"]))
        scores[key] = {"entail": float(got["entailment"]),
                       "contradict": float(got["contradiction"]),
                       "neutral": float(got["neutral"])}
    except Exception as exc:                      # recorded, never swallowed
        errors.append((key, repr(exc)))
    if n % 10 == 0 or n == len(PAIRS):
        el = time.time() - t0
        print(f"  {n}/{len(PAIRS)}  {el:5.1f}s elapsed, "
              f"~{el / n * (len(PAIRS) - n):5.1f}s left")

print(f"\nscored {len(scores)} of {len(PAIRS)} pairs in {time.time() - t0:.1f}s")
if errors:
    print(f"\n!! {len(errors)} pairs FAILED — do not download a partial file:")
    for k, e in errors[:10]:
        print("  ", k, e)

In [ ]:
# Cell 6 — completeness, then write and download.
#
# The apply script on the Mac refuses to run on a partial file, because an unscored
# pair would be dropped by phase3d's per-passage `except: continue` — silently
# lowering the max and moving the verdict for a reason that has nothing to do with
# the cache key. So the check happens here too, where it is cheap to fix by re-running
# the cell above.
missing = [k for _, _, k in PAIRS if k not in scores]
assert not missing, (f"{len(missing)} pairs unscored, e.g. {missing[:3]}. "
                     "Re-run the scoring cell; do not download this file.")

for k, v in scores.items():
    assert 0.0 <= v["entail"] <= 1.0 and 0.0 <= v["contradict"] <= 1.0, (k, v)

OUTNAME = "nli_rescore_scores_%s.json" % META["set"]
payload = {
    "meta": {
        "task": META["task"],
        "set": META["set"],
        "worklist": WORKLIST,
        "model": "facebook/bart-large-mnli",
        "device": DEVICE,
        "device_note": "-1 == CPU, matching phase3d.get_nli()",
        "transformers": transformers.__version__,
        "torch": torch.__version__,
        "pairs": len(scores),
        "rows": len(ITEMS),
        "candidate_labels": LABELS,
        "hypothesis_template_prefix": PREFIX,
        "key_format": META["key_format"]["fixed"],
        "gemini_key_present": GEMINI_KEY_PRESENT,
    },
    "scores": scores,
}
with open(OUTNAME, "w", encoding="utf-8") as fh:
    json.dump(payload, fh, indent=1)

print("wrote", OUTNAME)
print()
# A first look, before leaving Colab: did the collision actually break?
# Within a group the recorded scores were identical to four decimals. If the fresh
# scores still are, something is wrong with the work-list, not with the model.
import collections as _c
groups = _c.defaultdict(list)
for it in ITEMS:
    groups[it["collision_group"]].append(it)
print("per-group spread of the FRESH best-entail (recorded groups were identical):")
for g, its in sorted(groups.items(), key=lambda kv: -len(kv[1])):
    vals = []
    for it in its:
        e = max(scores[p["fixed_key"]]["entail"] for p in it["passages"])
        vals.append((it["claim_id"], round(e, 4), float(it["recorded"]["entail"] or 0)))
    spread = max(v for _, v, _ in vals) - min(v for _, v, _ in vals)
    flag = "  <-- STILL IDENTICAL, investigate" if spread < 1e-6 else ""
    print(f"  n={len(its)} spread={spread:.4f}{flag}")
    for cid, new, old in vals:
        print(f"      {cid:<14} recorded {old:.4f}  ->  fresh {new:.4f}")

files.download(OUTNAME)

## Back on the Mac

```bash
cd "/Users/sahibabc/8th Semester Materials/ThesisCode_copy/.claude/worktrees/rq2-batch3-table"
V="/Users/sahibabc/8th Semester Materials/ThesisCode_copy/.venv/bin/python"

# offline, no model, no network, a few seconds
"$V" current/nli_rescore_apply.py --set both \
     --scores ~/Downloads/nli_rescore_scores_both.json
```

That writes a **new** CSV — `current/nli_rescore_both_results.csv` — and prints:

* every row whose `(entail, contradict)` changed, and every 3d / 3e verdict that moved;
* **which member of each collision group wrote the cache entry and which inherited it**
  — the run order the cache never recorded. Whoever's score moves is the inheritor;
* the change in FALSE predictions, reported separately for the negatives (the TPR
  numerator) and the natural rows (the FPR numerator), never pooled.

The quoted baselines are not touched. `nonrouted_baseline_results.csv` and
`nonrouted_negatives_results.csv` stay exactly as they are — adopting the rescored
numbers is a separate, deliberate step, and it needs
`current/negatives_cache_audit.py` re-run to move any *published* TPR/FPR/J, because
those have denominators over the full 68 and 313, not over these 26 rows.

Before the real run you can prove the whole path end to end without Colab:

```bash
"$V" current/nli_rescore_apply.py --set both --simulate   # must report ZERO movement
"$V" current/nli_rescore_apply.py --selftest              # 16 checks
```

`--simulate` feeds the *recorded* scores back through the new code path. It must
reproduce the recorded verdicts exactly — that is what licenses reading any movement
in the real run as caused by the scores rather than by the apply script.